In [1]:
import pandas as pd
import pathlib
import numpy as np

In [2]:
INPUT_DIR = pathlib.Path('M:/urban_modeling/baus/BAUS Inputs')
#INPUT_DIR = pathlib.Path('/Volumes/Data/Models/urban_modeling/baus/BAUS Inputs')


### Final Blueprint minus H6 and Final Blueprint minus H8

In [3]:
# separate H6 and H8 projects
H6_H8_projects_file = INPUT_DIR / 'plan_strategies' / 'H6_H8_development_pipeline_entries_FBP_MAR.csv'
H6_H8_projects_df = pd.read_csv(H6_H8_projects_file)
print(H6_H8_projects_df.shape)
print(H6_H8_projects_df['source'].value_counts())
display(H6_H8_projects_df.head())

(1287, 30)
source
pub            940
mall_office    346
public           1
Name: count, dtype: int64


,Unnamed: 0,building_name,building_type_det,building_type,building_type_id,development_type_id,building_sqft,non_residential_sqft,residential_units,unit_ave_sqft,...,x,y,geom_id,year_built,tenure,stories,source,edit_date,editor,action_orig
0,0,NaN,MR,MR,12,5,224998,33500,116,0,...,-122.276390,37.794811,1.044829e+13,2045,rent,0,mall_office,20240122,ms,build
1,1,NaN,MR,MR,12,5,2971630,14000,1793,0,...,-122.277000,37.794147,5.349556e+11,2045,rent,0,mall_office,20240122,ms,add
2,2,NaN,MR,MR,12,5,1002969,180000,499,0,...,-122.273933,37.793633,1.767541e+12,2045,rent,0,mall_office,20240122,ms,build
3,3,NaN,MR,MR,12,5,657803,217924,267,0,...,-122.268257,37.791069,1.095146e+13,2045,rent,0,mall_office,20240122,ms,build
4,4,NaN,MR,MR,12,5,102711,26002,46,0,...,-122.279808,37.796997,1.537398e+13,2045,rent,0,mall_office,20240122,ms,build


In [4]:
H6_projects_df = H6_H8_projects_df[H6_H8_projects_df['source'] == 'mall_office']
H8_projects_df = H6_H8_projects_df[H6_H8_projects_df['source'].isin(['pub', 'public'])]

assert(H6_projects_df['residential_units'].sum() + H8_projects_df['residential_units'].sum() == H6_H8_projects_df['residential_units'].sum())

In [5]:
H6_projects_df.to_csv(INPUT_DIR / 'plan_strategies' / 'H6_development_pipeline_entries_FBP_MAR.csv', index=False)
H8_projects_df.to_csv(INPUT_DIR / 'plan_strategies' / 'H8_development_pipeline_entries_FBP_MAR.csv', index=False)

### Final Blueprint minus H3/EC4/EN4

#### 1: check if FBP parcels_geography.csv is compatible with NoProject zoningmods

In [6]:
parcels_geography_fbp_file = INPUT_DIR / 'basis_inputs' / 'crosswalks' / 'fbp_urbansim_parcel_classes_ot50pct_feb25_rwc_update_2025.csv'
parcels_geography_dbp_file = INPUT_DIR / 'basis_inputs' / 'crosswalks' / 'parcels_geography_2024_02_14.csv'
parcels_geography_np_file = INPUT_DIR / 'basis_inputs' / 'crosswalks' / 'parcels_geography_NP_2024_04_29.csv'

zoning_mods_fbp_file = INPUT_DIR / 'plan_strategies' / 'zoning_mods_PBA50Plus_FBP_pgfeb14_2025_ppafix.csv'
zoning_mods_np_file = INPUT_DIR / 'plan_strategies' / 'zoning_mods_PBA50Plus_NP_exp.csv'

zoningmodcat_cols_fbp = ["gg_id", "exd_id", "tra_id", "hra_id", "ppa_id", "ugb_id"]
zoningmodcat_cols_np = ["gg_id", "exd_id", "tra_id", "hra_id", "ppa_id", "ugb_id"]

parcels_geography_cols_fbp = ["gg_id", "exd_id", "pda_id", "tra_id", "hra_id", "dis_id", "ppa_id", "ugb_id"]
parcels_geography_cols_np =  ["gg_id", "exd_id", "pda_id", "tra_id", "hra_id", "epc_id", "dis_id", "ppa_id", "ugb_id"]

In [7]:
parcels_geography_fbp = pd.read_csv(parcels_geography_fbp_file, na_values=['NA','nan','NaN'])
parcels_geography_np = pd.read_csv(parcels_geography_np_file, na_values=['NA','nan','NaN'])
parcels_geography_np.rename({'PARCEL_ID': 'parcel_id'}, axis=1, inplace=True)

parcels_geography_dbp = pd.read_csv(parcels_geography_dbp_file, na_values=['NA','nan','NaN'])

print('parcels_geography_fbp rows:', parcels_geography_fbp.shape[0])
print('parcels_geography_np rows:', parcels_geography_np.shape[0])
print('parcels_geography_dbp rows:', parcels_geography_dbp.shape[0])

zoning_mods_fbp = pd.read_csv(zoning_mods_fbp_file)
zoning_mods_np = pd.read_csv(zoning_mods_np_file)

parcels_geography_fbp rows: 1956207
parcels_geography_np rows: 1956208
parcels_geography_dbp rows: 1956208


In [8]:
display(parcels_geography_fbp[['parcel_id'] + zoningmodcat_cols_fbp].head())
display(parcels_geography_np[['parcel_id'] + zoningmodcat_cols_np].head())

,parcel_id,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id
0,22.0,GG,NaN,NaN,NaN,NaN,UGB
1,58.0,GG,NaN,NaN,NaN,PPA,UGB
2,128.0,GG,NaN,NaN,NaN,NaN,UGB
3,217.0,GG,NaN,tra_4,NaN,NaN,UGB
4,358.0,GG,NaN,tra_4,NaN,NaN,UGB


,parcel_id,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id
0,229116,NaN,NaN,NaN,NaN,NaN,UGB
1,244166,NaN,NaN,NaN,NaN,NaN,UGB
2,202378,NaN,exd,NaN,NaN,NaN,UGB
3,2004420,NaN,NaN,NaN,NaN,NaN,NaN
4,340332,NaN,NaN,NaN,NaN,NaN,UGB


In [9]:
for col in set(zoningmodcat_cols_fbp + parcels_geography_cols_fbp):
    parcels_geography_fbp[col] = parcels_geography_fbp[col].astype(str).str.lower()
    parcels_geography_fbp[col] = parcels_geography_fbp[col].fillna('nan')

for col in set(zoningmodcat_cols_np + parcels_geography_cols_np):
    parcels_geography_np[col] = parcels_geography_np[col].astype(str).str.lower()
    parcels_geography_np[col] = parcels_geography_np[col].fillna('nan')

In [10]:
# FBP version has more parcels in GG than NP; DBP and NP have the same number of parcels in GG, because more parcels
# became GG due to transit projects (only added in Final Blueprint)
print(parcels_geography_fbp['gg_id'].value_counts(dropna=False))
print(parcels_geography_dbp['gg_id'].value_counts(dropna=False))
print(parcels_geography_np['gg_id'].value_counts(dropna=False))


gg_id
nan    1279970
gg      676237
Name: count, dtype: int64
gg_id
NaN    1358754
GG      597454
Name: count, dtype: int64
gg_id
nan    1358754
gg      597454
Name: count, dtype: int64


In [11]:
print('compare ugb_id')
print(parcels_geography_fbp['ugb_id'].value_counts(dropna=False))
print(parcels_geography_dbp['ugb_id'].value_counts(dropna=False))
print(parcels_geography_np['ugb_id'].value_counts(dropna=False))

compare ugb_id
ugb_id
ugb    1836513
nan     119694
Name: count, dtype: int64
ugb_id
UGB          1669191
Uninc UGB     167059
NaN           119958
Name: count, dtype: int64
ugb_id
ugb          1669191
uninc ugb     167059
nan            93315
exp1           10180
exp3_au         7297
exp1_np         4879
expmax_np       2573
exp2_np          880
exp0             783
exp0_np           51
Name: count, dtype: int64


In [12]:
# check if parcels_geography_fbp and parcels_geography_np can generate effectively same upzoning setting

# NoProject zoningmods applied to parcels_geography_fbp
parcels_geography_fbp_test = parcels_geography_fbp.copy()
parcels_geography_fbp_test["zoningmodcat"] = (
            parcels_geography_fbp_test[zoningmodcat_cols_fbp]
            .astype(str)
            .apply(lambda x: "".join(x), axis=1)
            .str.lower() 
        )
v_fbp = pd.merge(
    parcels_geography_fbp_test[['parcel_id','zoningmodcat']],
    zoning_mods_np,
    on='zoningmodcat',
    how='left'
)

# NoProject zoningmods applied to parcels_geography_np
parcels_geography_np_test = parcels_geography_np.copy()
parcels_geography_np_test["zoningmodcat"] = (
            parcels_geography_np_test[zoningmodcat_cols_np]
            .astype(str)
            .apply(lambda x: "".join(x), axis=1)
            .str.lower() 
        )
v_NP = pd.merge(
    parcels_geography_np_test[['parcel_id','zoningmodcat']],
    zoning_mods_np,
    on='zoningmodcat',
    how='left'
)

print('dua_up comparison:')
print(v_fbp['dua_up'].value_counts(dropna=False))
print(v_NP['dua_up'].value_counts(dropna=False))

print('dua_down comparison:')
print(v_fbp['dua_down'].value_counts(dropna=False))
print(v_NP['dua_down'].value_counts(dropna=False))

print('add_bldg comparison:')
print(v_fbp['add_bldg'].value_counts(dropna=False))
print(v_NP['add_bldg'].value_counts(dropna=False))

print('far_up comparison:')
print(v_fbp['far_up'].value_counts(dropna=False))
print(v_NP['far_up'].value_counts(dropna=False))

print('far_down comparison:')
print(v_fbp['far_down'].value_counts(dropna=False))
print(v_NP['far_down'].value_counts(dropna=False))

print('ugb_id comparison:')
print(v_fbp['ugb_id'].value_counts(dropna=False))
print(v_NP['ugb_id'].value_counts(dropna=False))


dua_up comparison:
dua_up
NaN    1956207
Name: count, dtype: int64
dua_up
NaN    1762506
5.5     193702
Name: count, dtype: int64
dua_down comparison:
dua_down
NaN    1956207
Name: count, dtype: int64
dua_down
NaN    1956208
Name: count, dtype: int64
add_bldg comparison:
add_bldg
NaN    1956207
Name: count, dtype: int64
add_bldg
NaN    1762506
HS      193702
Name: count, dtype: int64
far_up comparison:
far_up
NaN    1956207
Name: count, dtype: int64
far_up
NaN    1956208
Name: count, dtype: int64
far_down comparison:
far_down
NaN    1956207
Name: count, dtype: int64
far_down
NaN    1956208
Name: count, dtype: int64
ugb_id comparison:
ugb_id
NaN    1227646
UGB     728561
Name: count, dtype: int64
ugb_id
UGB          1669191
inun          167059
NaN            93315
exp1           10180
exp3_au         7297
exp1_np         4879
expmax_np       2573
exp2_np          880
exp0             783
exp0_np           51
Name: count, dtype: int64


##### These comparison show that if we use fbp version of parcels_geography to run FBP minus H3/EC4/EN4, even if we use NoProject zoningmods, we would stil miss the UGB expansion assumption we had for a scenario without the EN4 UGB strategy.

#### 2: create parcels_geography and zoningmods input

In [13]:
# Therefore, need to create a pair of parcels_geography and zoning_mods.
# Assumptions:
# - 1) follow Final Blueprint for all parcel designations (gg_id, tra_id, hra_id, etc.) except for ugb_id, so that the other designations can still be used in other strategies
# - 2)apply the principles of NoProject zonigmods, including upzoning in UGB-expansion areas and no zoning modifications elsewhere

# 1) parcels_geography
parcels_geography_FBP_withUGBexpansion_df = pd.merge(
    parcels_geography_fbp.loc[:, parcels_geography_fbp.columns != 'ugb_id'],
    parcels_geography_np.loc[:, ['parcel_id', 'ugb_id']],
    on='parcel_id',
    how='left'
)
print(parcels_geography_FBP_withUGBexpansion_df.shape[0])
print(parcels_geography_FBP_withUGBexpansion_df['ugb_id'].value_counts(dropna=False))

1956207
ugb_id
ugb          1669191
uninc ugb     167059
nan            93314
exp1           10180
exp3_au         7297
exp1_np         4879
expmax_np       2573
exp2_np          880
exp0             783
exp0_np           51
Name: count, dtype: int64


In [14]:
# verify it has all the fields needed by other FBP strategies:
for i in parcels_geography_cols_fbp:
    if not i in parcels_geography_FBP_withUGBexpansion_df:
        print(f"Column {i} is missing in parcels_geography_FBP_withUGBexpansion_df")

# verify tra_id categories
print(parcels_geography_fbp['tra_id'].value_counts(dropna=False))
print(parcels_geography_FBP_withUGBexpansion_df['tra_id'].value_counts(dropna=False))

tra_id
nan      1329969
tra_4     254188
tra_5     137561
tra_6     107582
tra_3      61962
tra_1      34343
tra_2      30602
Name: count, dtype: int64
tra_id
nan      1329969
tra_4     254188
tra_5     137561
tra_6     107582
tra_3      61962
tra_1      34343
tra_2      30602
Name: count, dtype: int64


In [15]:
# validate columns
print(parcels_geography_FBP_withUGBexpansion_df.columns.sort_values())
print(parcels_geography_fbp.columns.sort_values())
assert(set(parcels_geography_FBP_withUGBexpansion_df.columns) == set(parcels_geography_fbp.columns))

Index(['dis_id', 'exd_id', 'exp2020_id', 'exsfd_id', 'gg_id', 'hra_id',
       'parcel_id', 'pda_id', 'ppa_id', 'tpp_id', 'tra_id', 'ugb_id'],
      dtype='str')
Index(['dis_id', 'exd_id', 'exp2020_id', 'exsfd_id', 'gg_id', 'hra_id',
       'parcel_id', 'pda_id', 'ppa_id', 'tpp_id', 'tra_id', 'ugb_id'],
      dtype='str')


In [16]:
# write out
parcels_geography_FBP_withUGBexpansion_df.to_csv(
    INPUT_DIR / 'basis_inputs' / 'crosswalks' / 'parcels_geography_FBP_withUGBexpansion.csv', index=False)

In [17]:
# 2) create zoning_mods following the methodology in \bayarea_urbansim\scripts\strategy_building\build_PBA50Plus_NP_zoningmods_exp.ipynb

# add a zoningmodcat column to parcels geography
parcels_geography_FBP_withUGBexpansion_df['zoningmodcat'] = ''
for col in zoningmodcat_cols_fbp:
    # leave this line in only because the PBA50 BAUS code makes strings lowercase
    # and it will ensure the PBA50 strategy inputs will work
    parcels_geography_FBP_withUGBexpansion_df[col] = parcels_geography_FBP_withUGBexpansion_df[col].astype(str).str.lower()
    parcels_geography_FBP_withUGBexpansion_df['zoningmodcat'] = parcels_geography_FBP_withUGBexpansion_df['zoningmodcat'] + parcels_geography_FBP_withUGBexpansion_df[col]

print(parcels_geography_FBP_withUGBexpansion_df['zoningmodcat'].unique())


<ArrowStringArray>
[        'ggnannannannanugb',         'ggnannannanppaugb',
       'ggnantra_4nannanugb',       'ggnantra_1nannanugb',
       'ggexdtra_1nannanugb',       'ggexdtra_4nannanugb',
       'ggnantra_6nannanugb',       'ggexdtra_6nannanugb',
       'ggnantra_4hrananugb',        'nanexdnanhrananugb',
 ...
         'ggexdnanhrappaugb',       'ggnantra_6nanppaugb',
    'nanexdnanhrananexp3_au', 'ggnantra_2hrananuninc ugb',
   'ggnantra_5nannanexp3_au',        'ggnannannanppaexp0',
        'ggnannanhrananexp1',       'ggexdtra_3nannannan',
       'ggnantra_3nannannan',         'ggexdnannanppanan']
Length: 147, dtype: str


In [18]:
# build zoningmods using zoningmodcat values from parcels geography as the primary key
zoningmods = parcels_geography_FBP_withUGBexpansion_df.groupby(['zoningmodcat']+zoningmodcat_cols_fbp).size().reset_index(name='count')

# then add other zoningmods columns
string_cols = [
    # "gg_id", "exd_id", "tra_id", "hra_id", "ppa_id", "ugb_id",
    "add_bldg", "drop_bldg",
]
numeric_cols = ["dua_up", "far_up", "dua_down", "far_down"]

for col in string_cols:
    zoningmods[col] = pd.Series(pd.NA, index=zoningmods.index, dtype="object")

for col in numeric_cols:
    zoningmods[col] = np.nan

zoningmods.loc[zoningmods.zoningmodcat.str.contains("uninc ugb"), "ugb_id"] = "inun"

# Add Zoning Step 1: Add unincorporated UGB zoning- Applied to No Project but not Draft Blueprint
# additional upzoning is added for historic expansion
zoningmods.loc[(zoningmods['ugb_id'].isin(['exp0', 'exp0_np', 'exp1', 'exp1_np', 'exp2_np', 'exp3_au', 'expmax_np', 'inun'])) , 'dua_up'] = 5.5
zoningmods.loc[(zoningmods['ugb_id'].isin(['exp0', 'exp0_np', 'exp1', 'exp1_np', 'exp2_np', 'exp3_au', 'expmax_np', 'inun'])) , 'add_bldg'] = 'HS'


In [19]:
display(zoningmods)
display(zoningmods.loc[zoningmods['dua_up'] == 5.5])

zoningmods_PlanMinusH3EC4EN4 = zoningmods.copy()
zoningmods_PlanMinusH3EC4EN4.loc[:,zoningmodcat_cols_fbp] = zoningmods_PlanMinusH3EC4EN4.loc[:,zoningmodcat_cols_fbp].replace('nan',np.nan)

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,add_bldg,drop_bldg,dua_up,far_up,dua_down,far_down
0,ggexdnanhrananexp1,gg,exd,nan,hra,nan,exp1,11,HS,<NA>,5.5,NaN,NaN,NaN
1,ggexdnanhranannan,gg,exd,nan,hra,nan,nan,1,<NA>,<NA>,NaN,NaN,NaN,NaN
2,ggexdnanhrananugb,gg,exd,nan,hra,nan,ugb,5585,<NA>,<NA>,NaN,NaN,NaN,NaN
3,ggexdnanhrananuninc ugb,gg,exd,nan,hra,nan,inun,205,HS,<NA>,5.5,NaN,NaN,NaN
4,ggexdnanhrappaugb,gg,exd,nan,hra,ppa,ugb,1,<NA>,<NA>,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,nannannannannanexp3_au,nan,nan,nan,nan,nan,exp3_au,2226,HS,<NA>,5.5,NaN,NaN,NaN
143,nannannannannanexpmax_np,nan,nan,nan,nan,nan,expmax_np,1208,HS,<NA>,5.5,NaN,NaN,NaN
144,nannannannannannan,nan,nan,nan,nan,nan,nan,37798,<NA>,<NA>,NaN,NaN,NaN,NaN
145,nannannannannanugb,nan,nan,nan,nan,nan,ugb,56402,<NA>,<NA>,NaN,NaN,NaN,NaN


,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,add_bldg,drop_bldg,dua_up,far_up,dua_down,far_down
0,ggexdnanhrananexp1,gg,exd,nan,hra,nan,exp1,11,HS,<NA>,5.5,NaN,NaN,NaN
3,ggexdnanhrananuninc ugb,gg,exd,nan,hra,nan,inun,205,HS,<NA>,5.5,NaN,NaN,NaN
5,ggexdnannannanexp0,gg,exd,nan,nan,nan,exp0,38,HS,<NA>,5.5,NaN,NaN,NaN
6,ggexdnannannanexpmax_np,gg,exd,nan,nan,nan,expmax_np,2,HS,<NA>,5.5,NaN,NaN,NaN
9,ggexdnannannanuninc ugb,gg,exd,nan,nan,nan,inun,2581,HS,<NA>,5.5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,nannannannannanexp1_np,nan,nan,nan,nan,nan,exp1_np,1916,HS,<NA>,5.5,NaN,NaN,NaN
141,nannannannannanexp2_np,nan,nan,nan,nan,nan,exp2_np,326,HS,<NA>,5.5,NaN,NaN,NaN
142,nannannannannanexp3_au,nan,nan,nan,nan,nan,exp3_au,2226,HS,<NA>,5.5,NaN,NaN,NaN
143,nannannannannanexpmax_np,nan,nan,nan,nan,nan,expmax_np,1208,HS,<NA>,5.5,NaN,NaN,NaN


In [20]:
# export
zoningmods_PlanMinusH3EC4EN4.to_csv(r"M:\urban_modeling\baus\BAUS Inputs\plan_strategies\zoning_mods_PBA50plus_PlanMinusH3EC4EN4.csv", index=False)

In [21]:
# also create a version for a scenario with FBP upzoning but NP treatment for UGB (aka "FBP without Strategy EN4")
# start with the above zoningmods and add H3/EC4 upzoning, based on \bayarea_urbansim\scripts\strategy_building\build_BPAPlus_FBP_zoningmods.ipynb

# Step 2: Add residential and non-residential growth geography upzoning, which can override unincorporated UGB zoning

# First we define the universe: Growth Geographies (GG) with different area classifications
# - TRA: Transit-Rich Area (TRA1, TRA2, TRA3, TRA4, TRA5, TRA6)
# - HRA: High Resource Areas 
# - EXD: Residential Parcel (exd_id == 'exd' means it is a residential parcel)
# create a df with unique combinations of the components - and zoningmodcat

# Define common expressions for filtering
is_gg = "gg_id == 'gg'"
is_tra1 = "tra_id == 'tra_1'"
is_tra2 = "tra_id == 'tra_2'"
is_tra3 = "tra_id == 'tra_3'"
is_tra4 = "tra_id == 'tra_4'"
is_tra5 = "tra_id == 'tra_5'"
is_tra6 = "tra_id == 'tra_6'"
is_hra = "hra_id == 'hra'"
is_not_hra = "hra_id != 'hra'"
is_exd = "exd_id == 'exd'"
is_not_exd = "exd_id != 'exd'"
is_tra = "tra_id != 'nan'"
# is_no_tra = "tra_id=='nan'"
# is_no_hra = "hra_id=='nan'"
is_ppa = "ppa_id=='ppa'"
# is_ugb = "ugb_id=='ugb'"

# # List of building types to add for least restrictive areas
building_list_inclusive = 'HT HM OF HO SC RS RB MR MT ME'

# # list of residential types to add for residential areas
building_list_res = 'HM MR'

# list of columns to update for nonres, res sets
non_res_cols = ['dua_up', 'far_up', 'add_bldg']
res_cols = ['dua_up', 'add_bldg']

# Apply updates based on conditions

# initialze the three colums to set
zoningmods['drop_bldg'] =''

# Apply updates using masks

# GG + TRA1 + HRA
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra1} & {is_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 300, 30
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra1} & {is_hra} & {is_exd}"), ['dua_up']] = 50 

# GG + TRA1 (no HRA)
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra1} & {is_not_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 250, 30
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra1} & {is_not_hra} & {is_exd}"), ['dua_up']] = 50 

# GG + TRA2 + HRA
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra2} & {is_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 250, 25
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra2} & {is_hra} & {is_exd}"), ['dua_up']] = 50 

# GG + TRA2 (no HRA)
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra2} & {is_not_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 200, 25
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra2} & {is_not_hra} & {is_exd}"), ['dua_up']] = 35 

# GG + TRA3 + HRA
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra3} & {is_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 200, 15
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra3} & {is_hra} & {is_exd}"), ['dua_up']] = 50

# GG + TRA3 (no HRA)
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra3} & {is_not_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 150, 15
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra3} & {is_not_hra} & {is_exd}"), ['dua_up']] = 30

# GG + TRA4 + HRA
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra4} & {is_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 100, 5
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra4} & {is_hra} & {is_exd}"), ['dua_up']] = 50 

# GG + TRA4 (no HRA)
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra4} & {is_not_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 100, 5  
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra4} & {is_not_hra} & {is_exd}"), ['dua_up']] = 30 

# GG + TRA5 + HRA
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra5} & {is_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 75, 2.5  
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra5} & {is_hra} & {is_exd}"), ['dua_up']] = 30

# GG + TRA5 (no HRA)
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra5} & {is_not_hra} & {is_not_exd}"), ['dua_up', 'far_up']] = 50, 2.5  
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra5} & {is_not_hra} & {is_exd}"), ['dua_up']] = 25  

# GG + TRA6 
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra6} & {is_not_exd}"), ['dua_up', 'far_up']] = 35, 0  
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_tra6} & {is_exd}"), ['dua_up']] = 20 

# add building types
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_not_exd} & {is_tra}"), ['add_bldg']] = building_list_inclusive
zoningmods.loc[zoningmods.eval(f"{is_gg} & {is_exd} & {is_tra}"), ['add_bldg']] = building_list_res


In [22]:
# Step 3: Add PPA zoning, which can override unincorporated UGB zoning and growth geography upzoning

# GG + PPA  (NEW - Corrected values and conditions)
zoningmods.loc[zoningmods.eval(f"{is_ppa} & {is_not_exd}"), ['far_up']] =  0.35
zoningmods.loc[zoningmods.eval(f"{is_ppa} & {is_not_exd}"), ['drop_bldg']] = building_list_inclusive
zoningmods.loc[zoningmods.eval(f"{is_ppa} & {is_not_exd}"), ['add_bldg']] = 'IW'
zoningmods.loc[zoningmods.eval(f"{is_ppa} & {is_exd}"), ['far_up']] =  0
zoningmods.loc[zoningmods.eval(f"{is_ppa} & {is_exd}"), ['drop_bldg']] = 'HT HM OF HO SC RS RB MR MT ME IW'

In [23]:
zoningmods_PlanMinusEN4 = zoningmods.copy()
zoningmods_PlanMinusEN4.loc[:,zoningmodcat_cols_fbp] = zoningmods_PlanMinusEN4.loc[:,zoningmodcat_cols_fbp].replace('nan',np.nan)
display(zoningmods_PlanMinusEN4)

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,add_bldg,drop_bldg,dua_up,far_up,dua_down,far_down
0,ggexdnanhrananexp1,gg,exd,NaN,hra,NaN,exp1,11,HS,,5.5,NaN,NaN,NaN
1,ggexdnanhranannan,gg,exd,NaN,hra,NaN,NaN,1,<NA>,,NaN,NaN,NaN,NaN
2,ggexdnanhrananugb,gg,exd,NaN,hra,NaN,ugb,5585,<NA>,,NaN,NaN,NaN,NaN
3,ggexdnanhrananuninc ugb,gg,exd,NaN,hra,NaN,inun,205,HS,,5.5,NaN,NaN,NaN
4,ggexdnanhrappaugb,gg,exd,NaN,hra,ppa,ugb,1,<NA>,HT HM OF HO SC RS RB MR MT ME IW,NaN,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,nannannannannanexp3_au,NaN,NaN,NaN,NaN,NaN,exp3_au,2226,HS,,5.5,NaN,NaN,NaN
143,nannannannannanexpmax_np,NaN,NaN,NaN,NaN,NaN,expmax_np,1208,HS,,5.5,NaN,NaN,NaN
144,nannannannannannan,NaN,NaN,NaN,NaN,NaN,NaN,37798,<NA>,,NaN,NaN,NaN,NaN
145,nannannannannanugb,NaN,NaN,NaN,NaN,NaN,ugb,56402,<NA>,,NaN,NaN,NaN,NaN


In [24]:
display(zoningmods_PlanMinusEN4.loc[zoningmods_PlanMinusEN4['dua_up'] == 5.5])

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,add_bldg,drop_bldg,dua_up,far_up,dua_down,far_down
0,ggexdnanhrananexp1,gg,exd,NaN,hra,NaN,exp1,11,HS,,5.5,NaN,NaN,NaN
3,ggexdnanhrananuninc ugb,gg,exd,NaN,hra,NaN,inun,205,HS,,5.5,NaN,NaN,NaN
5,ggexdnannannanexp0,gg,exd,NaN,NaN,NaN,exp0,38,HS,,5.5,NaN,NaN,NaN
6,ggexdnannannanexpmax_np,gg,exd,NaN,NaN,NaN,expmax_np,2,HS,,5.5,NaN,NaN,NaN
9,ggexdnannannanuninc ugb,gg,exd,NaN,NaN,NaN,inun,2581,HS,,5.5,NaN,NaN,NaN
12,ggexdnannanppauninc ugb,gg,exd,NaN,NaN,ppa,inun,35,HS,HT HM OF HO SC RS RB MR MT ME IW,5.5,0.00,NaN,NaN
49,ggnannanhrananexp1,gg,NaN,NaN,hra,NaN,exp1,1,HS,,5.5,NaN,NaN,NaN
52,ggnannanhrananuninc ugb,gg,NaN,NaN,hra,NaN,inun,2,HS,,5.5,NaN,NaN,NaN
54,ggnannannannanexp0,gg,NaN,NaN,NaN,NaN,exp0,27,HS,,5.5,NaN,NaN,NaN
55,ggnannannannanexp1,gg,NaN,NaN,NaN,NaN,exp1,2,HS,,5.5,NaN,NaN,NaN


In [25]:
# write out
zoningmods_PlanMinusEN4.to_csv(r"M:\urban_modeling\baus\BAUS Inputs\plan_strategies\zoning_mods_PBA50plus_PlanMinusEN4.csv", index=False)

In [26]:
# additional debug
v_NP['dua_up'].value_counts()

dua_up
5.5    193702
Name: count, dtype: int64

In [27]:
zoningmods_PlanMinusH3EC4EN4.groupby('dua_up')['count'].sum()

dua_up
5.5    193702
Name: count, dtype: int64

In [28]:
zoningmods_PlanMinusEN4.groupby('dua_up')['count'].sum()

dua_up
5.5      165101
20.0      91068
25.0      53525
30.0     224240
35.0      31580
50.0     165389
75.0       7392
100.0     28238
150.0      5949
200.0      8878
250.0      9808
300.0       171
Name: count, dtype: int64

In [29]:
zoning_mods_fbp.groupby('dua_up')['count'].sum()

dua_up
20.0      90636
25.0      53192
30.0     224167
35.0      31366
50.0     164991
75.0       7355
100.0     27890
150.0      5944
200.0      8878
250.0      9806
300.0       171
Name: count, dtype: int64

In [30]:
zoningmods_PlanMinusEN4.loc[zoningmods_PlanMinusEN4['dua_up'] == 250]

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,add_bldg,drop_bldg,dua_up,far_up,dua_down,far_down
67,ggnantra_1nannannan,gg,NaN,tra_1,NaN,NaN,NaN,3,HT HM OF HO SC RS RB MR MT ME,,250.0,30.0,NaN,NaN
68,ggnantra_1nannanugb,gg,NaN,tra_1,NaN,NaN,ugb,8024,HT HM OF HO SC RS RB MR MT ME,,250.0,30.0,NaN,NaN
69,ggnantra_1nannanuninc ugb,gg,NaN,tra_1,NaN,NaN,inun,81,HT HM OF HO SC RS RB MR MT ME,,250.0,30.0,NaN,NaN
70,ggnantra_2hranannan,gg,NaN,tra_2,hra,NaN,NaN,2,HT HM OF HO SC RS RB MR MT ME,,250.0,25.0,NaN,NaN
71,ggnantra_2hrananugb,gg,NaN,tra_2,hra,NaN,ugb,1696,HT HM OF HO SC RS RB MR MT ME,,250.0,25.0,NaN,NaN
72,ggnantra_2hrananuninc ugb,gg,NaN,tra_2,hra,NaN,inun,2,HT HM OF HO SC RS RB MR MT ME,,250.0,25.0,NaN,NaN


In [31]:
zoning_mods_fbp.loc[zoning_mods_fbp['dua_up'] == 250]

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,dua_up,far_up,add_bldg,yaml_category,drop_bldg,dua_down,far_down
38,ggnantra_1nannanugb,GG,NaN,tra_1,NaN,NaN,UGB,8106,250.0,30.0,HT HM OF HO SC RS RB MR MT ME,GG_TRA1_NonHRA_NonRes,NaN,NaN,NaN
39,ggnantra_2hrananugb,GG,NaN,tra_2,HRA,NaN,UGB,1700,250.0,25.0,HT HM OF HO SC RS RB MR MT ME,GG_TRA2_HRA_NonRes,NaN,NaN,NaN


In [32]:
zoningmods_PlanMinusEN4.loc[
    ((zoningmods_PlanMinusEN4['gg_id'] == 'gg') & (zoningmods_PlanMinusEN4['tra_id'] == 'tra_1') & (zoningmods_PlanMinusEN4['exd_id'] != 'exd') & (zoningmods_PlanMinusEN4['hra_id'] != 'hra')) | (
        (zoningmods_PlanMinusEN4['gg_id'] == 'gg') & (zoningmods_PlanMinusEN4['tra_id'] == 'tra_2') & (zoningmods_PlanMinusEN4['exd_id'] != 'exd') & (zoningmods_PlanMinusEN4['hra_id'] == 'hra')
    ) ]

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,add_bldg,drop_bldg,dua_up,far_up,dua_down,far_down
67,ggnantra_1nannannan,gg,NaN,tra_1,NaN,NaN,NaN,3,HT HM OF HO SC RS RB MR MT ME,,250.0,30.0,NaN,NaN
68,ggnantra_1nannanugb,gg,NaN,tra_1,NaN,NaN,ugb,8024,HT HM OF HO SC RS RB MR MT ME,,250.0,30.0,NaN,NaN
69,ggnantra_1nannanuninc ugb,gg,NaN,tra_1,NaN,NaN,inun,81,HT HM OF HO SC RS RB MR MT ME,,250.0,30.0,NaN,NaN
70,ggnantra_2hranannan,gg,NaN,tra_2,hra,NaN,NaN,2,HT HM OF HO SC RS RB MR MT ME,,250.0,25.0,NaN,NaN
71,ggnantra_2hrananugb,gg,NaN,tra_2,hra,NaN,ugb,1696,HT HM OF HO SC RS RB MR MT ME,,250.0,25.0,NaN,NaN
72,ggnantra_2hrananuninc ugb,gg,NaN,tra_2,hra,NaN,inun,2,HT HM OF HO SC RS RB MR MT ME,,250.0,25.0,NaN,NaN


In [33]:
zoning_mods_fbp.loc[
    ((zoning_mods_fbp['gg_id'] == 'GG') & (zoning_mods_fbp['tra_id'] == 'tra_1') & (zoning_mods_fbp['exd_id'] != 'exd') & (zoning_mods_fbp['hra_id'] != 'HRA')) | (
        (zoning_mods_fbp['gg_id'] == 'GG') & (zoning_mods_fbp['tra_id'] == 'tra_2') & (zoning_mods_fbp['exd_id'] != 'exd') & (zoning_mods_fbp['hra_id'] == 'HRA')
    ) ]

,zoningmodcat,gg_id,exd_id,tra_id,hra_id,ppa_id,ugb_id,count,dua_up,far_up,add_bldg,yaml_category,drop_bldg,dua_down,far_down
37,ggnantra_1nannannan,GG,NaN,tra_1,NaN,NaN,NaN,2,NaN,NaN,NaN,Outside_UGB_Zoning,NaN,0.0,0.0
38,ggnantra_1nannanugb,GG,NaN,tra_1,NaN,NaN,UGB,8106,250.0,30.0,HT HM OF HO SC RS RB MR MT ME,GG_TRA1_NonHRA_NonRes,NaN,NaN,NaN
39,ggnantra_2hrananugb,GG,NaN,tra_2,HRA,NaN,UGB,1700,250.0,25.0,HT HM OF HO SC RS RB MR MT ME,GG_TRA2_HRA_NonRes,NaN,NaN,NaN
